In [3]:
import json
import xml.etree.ElementTree as ET
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors

# ---------------------------------------------------------
# 0. Mapping ชื่อจังหวัดภาษาไทย (soil.xml) -> อังกฤษมาตรฐาน (climate_normal.xml)
# ---------------------------------------------------------
THAI_TO_ENG_PROVINCE = {
    "กำแพงเพชร": "KAMPHAENG PHET",
    "ตรัง": "TRANG",
    "ตาก": "TAK",
    "นครราชสีมา": "NAKHON RATCHASIMA",
    "นครสวรรค์": "NAKHON SAWAN",
    "ปทุมธานี": "PATHUM THANI",
    "พะเยา": "PHAYAO",
    "ราชบุรี": "RATCHABURI",
    "ร้อยเอ็ด": "ROI ET",
    "สงขลา": "SONGKHLA",
    "สุโขทัย": "SUKHOTHAI",
    "อุดรธานี": "UDON THANI",
    "อุทัยธานี": "UTHAI THANI",
    "อุบลราชธานี": "UBON RATCHATHANI",
    "เชียงราย": "CHIANG RAI",
    "เชียงใหม่": "CHIANG MAI",
    "เพชรบุรี": "PHETCHABURI",
    "แพร่": "PHRAE",
    "แม่ฮ่องสอน": "MAE HONG SON"
}

# แก้คำสะกด/เว้นวรรคพิเศษของชื่อสถานีใน climate_normal.xml ให้ตรงกับชื่อจังหวัดมาตรฐาน
STATION_ALIAS = {
    "RATCHABURI": "RATCHABURI",
    "RATCHA BURI": "RATCHABURI",
    "PATHUMTHANI": "PATHUM THANI",
    "PATHUM THANI": "PATHUM THANI",
    "CHAING RAI AGROMET.": "CHIANG RAI",
    "CHIANG RAI AGROMET.": "CHIANG RAI",
    "UBON RATCHATHANI AGROMET.": "UBON RATCHATHANI",
    "ROI ET AGROMET.": "ROI ET",
    "MAE JO AGROMET.": "CHIANG MAI",
    "PHETCHA BURI": "PHETCHABURI",
    "TRANG AIRPORT": "TRANG"
}

# ---------------------------------------------------------
# Step 1.1: อ่านและสรุปข้อมูลดินรายจังหวัดจาก soil.xml (โครงสร้าง JSON)
# ---------------------------------------------------------
def parse_soil_json(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        raw_data = json.load(f)
    
    df_soil_raw = pd.DataFrame(raw_data)
    df_soil_raw['value_num'] = pd.to_numeric(df_soil_raw['value'], errors='coerce')
    
    def map_param(p):
        if '(P)' in p:
            return 'P'
        elif '(K)' in p:
            return 'K'
        elif '(pH)' in p:
            return 'ph'
        elif '(OM)' in p:
            return 'OM'
        elif '(EC)' in p:
            return 'EC'
        return None

    df_soil_raw['feature'] = df_soil_raw['parameter'].apply(map_param)
    df_soil_raw = df_soil_raw.dropna(subset=['feature', 'value_num', 'province'])
    
    df_soil_raw['location'] = df_soil_raw['province'].str.strip().map(THAI_TO_ENG_PROVINCE)
    df_soil_raw = df_soil_raw.dropna(subset=['location'])
    
    df_soil_pivot = df_soil_raw.pivot_table(
        index='location',
        columns='feature',
        values='value_num',
        aggfunc='median'
    ).reset_index()
    
    return df_soil_pivot

# ---------------------------------------------------------
# Step 1.2: อ่านและสรุปข้อมูลสภาพอากาศรายจังหวัดจาก climate_normal.xml
# ---------------------------------------------------------
def parse_climate_xml(xml_file_path):
    tree = ET.parse(xml_file_path)
    root = tree.getroot()
    
    months = ['January', 'February', 'March', 'April', 'May', 'June', 
              'July', 'August', 'September', 'October', 'November', 'December']
    
    records = []
    for station in root.findall('StationClimateNormal'):
        st_raw = station.find('StationName').text.strip().replace('*', '').upper()
        st_clean = STATION_ALIAS.get(st_raw, st_raw)
        
        if st_clean not in set(THAI_TO_ENG_PROVINCE.values()):
            continue
            
        row_data = {'location': st_clean, 'is_main_station': (st_raw == st_clean)}
        for nv in station.findall('NormalValue'):
            elem_elem = nv.find('Element')
            if elem_elem is None or elem_elem.text is None:
                continue
            elem_name = elem_elem.text.strip()
            vals = [
                float(nv.find(f'ValueOn{m}').text)
                for m in months
                if nv.find(f'ValueOn{m}') is not None and nv.find(f'ValueOn{m}').text
            ]
            if not vals:
                continue
                
            if elem_name == 'Mean Dry-bulb Temperature':
                row_data['temperature'] = np.mean(vals)
            elif elem_name == 'Mean Relative Humidity':
                row_data['humidity'] = np.mean(vals)
            elif elem_name == 'Total Rainfall':
                row_data['rainfall'] = np.mean(vals)
        records.append(row_data)
        
    df_clim = pd.DataFrame(records)
    df_clim = df_clim.groupby(['location', 'is_main_station'], as_index=False).mean(numeric_only=True)
    df_clim = df_clim.sort_values(by=['location', 'is_main_station'], ascending=[True, False])
    df_clim = df_clim.drop_duplicates(subset=['location']).drop(columns=['is_main_station'])
    
    return df_clim

# ---------------------------------------------------------
# Step 1.3: รวมไฟล์ Soil + Climate เข้าด้วยกันตามจังหวัด (ข้อ 1)
# ---------------------------------------------------------
df_soil = parse_soil_json('../data/soil.xml')
df_climate = parse_climate_xml('../data/climate_normal.xml')

df_province_profile = pd.merge(df_soil, df_climate, on='location', how='left')

# เติมค่าว่าง (NaN) ในคอลัมน์ตัวเลขทั้งหมด ทั้งค่าดินและอากาศ ด้วยค่ามัธยฐาน (median)
numeric_cols = df_province_profile.select_dtypes(include=[np.number]).columns
for col in numeric_cols:
    df_province_profile[col] = df_province_profile[col].fillna(df_province_profile[col].median())

df_province_profile.to_csv('../data/province_soil_climate_merged.csv', index=False)
print("=== ข้อ 1: รวมข้อมูล Soil + Climate สำเร็จ ===")
print(f"จำนวนจังหวัดทั้งหมด: {len(df_province_profile)} จังหวัด | ค่าว่างที่เหลือ: {df_province_profile.isna().sum().sum()}")
print(df_province_profile[['location', 'P', 'K', 'ph', 'temperature', 'humidity', 'rainfall']].head(10))

# ---------------------------------------------------------
# Step 2: เทียบความใกล้เคียงกับ crop_recommendation_cleaned.csv เพื่อใส่ Location (ข้อ 2)
# ---------------------------------------------------------
df_crop = pd.read_csv('../data/crop_recommendation_cleaned.csv')

match_features = ['P', 'K', 'temperature', 'humidity', 'ph', 'rainfall']

scaler_match = StandardScaler()
crop_scaled = scaler_match.fit_transform(df_crop[match_features])
prov_scaled = scaler_match.transform(df_province_profile[match_features])

nn = NearestNeighbors(n_neighbors=1, metric='euclidean')
nn.fit(prov_scaled)
distances, indices = nn.kneighbors(crop_scaled)

df_enriched = df_crop.copy()
df_enriched['location'] = df_province_profile.iloc[indices.flatten()]['location'].values

df_enriched.to_csv('../data/enriched_crop_dataset.csv', index=False)

print("\n=== ข้อ 2: สร้างไฟล์ enriched_crop_dataset.csv สำเร็จ ===")
print(f"ขนาดข้อมูล: {df_enriched.shape[0]} แถว, {df_enriched.shape[1]} คอลัมน์")
print("\nจำนวนข้อมูลที่ถูกจับคู่เข้ากับแต่ละจังหวัด:")
print(df_enriched['location'].value_counts())

=== ข้อ 1: รวมข้อมูล Soil + Climate สำเร็จ ===
จำนวนจังหวัดทั้งหมด: 19 จังหวัด | ค่าว่างที่เหลือ: 0
            location     P      K    ph  temperature   humidity    rainfall
0         CHIANG MAI  28.0   33.0  5.60    25.825083  70.898417   94.214083
1         CHIANG RAI  12.0   66.0  4.87    24.450000  76.099333  140.876333
2     KAMPHAENG PHET  33.0   43.0  6.80    27.401167  76.242917  108.705000
3       MAE HONG SON  16.0  180.5  5.50    25.722250  74.029750  107.692500
4  NAKHON RATCHASIMA   3.0   39.0  6.20    27.349750  70.136083   89.282417
5       NAKHON SAWAN   9.5   15.5  6.90    28.216333  72.028917   95.798750
6       PATHUM THANI  10.0  369.0  4.10    28.730750  73.754000  116.385500
7             PHAYAO   7.0   68.0  5.52    25.146250  75.392750   94.760750
8        PHETCHABURI  90.5   94.0  7.30    27.919000  76.416500   82.275667
9              PHRAE   7.5   87.0  4.99    26.415667  75.611667   92.904167

=== ข้อ 2: สร้างไฟล์ enriched_crop_dataset.csv สำเร็จ ===
ขนาดข